# NB1 · The Before and the After

**AI Agents for Enterprises · Lecture 1 · Vizuara · Dr. Rajat Dandekar**

You are the engineer Rasoi Steel Works has hired. Rasoi makes pressure cookers in Pune and has run on
**Bahi ERP 7.2** since 1998. On Monday 26 October 2026 it needs 20,000 silicone gaskets (`GSK-SIL-05L`) from
Deccan Polymers before the Diwali run. (Everything here is fictional: the ERP, the plant, the people, the data.)

In this notebook you will:
1. meet Bahi from Python, as a **black box**: call a few services and read the envelope they return;
2. do the gasket order **the legacy way**, screen by screen, in code, and count every step;
3. let **Ask Rasoi** do the same missions, one sentence each, through the toolbox over **MCP**;
4. open a tool: its `tool.json` (what the LLM reads) and its `tool.py` (what runs), and call one yourself;
5. put the Before and the After side by side;
6. start **your module** for assignment A1.

The model calls run in **replay mode**: they come from our recorded run (cassette `L1v3_after`), so the notebook
runs with no API key and shows exactly what happened. Six exercises are marked `TODO (exercise N)`; each has its
solution in place behind `SOLUTION = True`, so the notebook runs top to bottom. Set it to `False` to try yourself.

In [1]:
# Setup: install the Bahi sandbox and its recorded model replies. Runs in about a minute on Colab.
# REPLAY mode (the default when you have no key) plays back our recorded model calls, so every cell runs
# without an API key and shows the numbers from our own run. To call a model live, set a key below.
import os, sys, subprocess
COURSE_REPO = "https://github.com/RajatDandekar/agents-for-enterprises"   # course code (Bahi sandbox + reference agent)
try:
    import bahi  # already available (local checkout or Modal)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"bahi-erp[all] @ git+{COURSE_REPO}.git#subdirectory=bahi"], check=True)
    import bahi
# os.environ["GEMINI_API_KEY"] = "..."        # optional: your own key, to run live
# os.environ["OPENROUTER_API_KEY"] = "..."    # optional: GPT, Claude, Llama ... via OpenRouter
os.environ.setdefault("BAHI_LLM_MODE", "auto")  # auto = replay what was recorded, call live only for new requests
print("Bahi", bahi.__version__, "· model mode:", os.environ["BAHI_LLM_MODE"])

Bahi 0.1.0 · model mode: replay


In [2]:
import json

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)


def inr(x):
    """Indian digit grouping: 280840 -> ₹2,80,840."""
    s = f"{abs(round(x)):d}"
    if len(s) > 3:
        head, tail = s[:-3], s[-3:]
        head = ",".join([head[max(0, i - 2):i] for i in range(len(head), 0, -2)][::-1])
        s = head + "," + tail
    return "₹" + s


def show_json(d, n=1200):
    print(json.dumps(d, indent=1, default=str, ensure_ascii=False)[:n])


print(inr(280840), inr(250000))

₹2,80,840 ₹2,50,000


## 1. Meet Bahi, as a black box

`ERP()` is one whole Rasoi in memory: items, suppliers, people, cost centres and past orders, with the clock at
Monday 26 October 2026. `serve()` puts it behind HTTP, the way Bahi's own legacy client reaches it.

A **service** is one operation of the ERP. An input goes in (a flat JSON payload), an output comes out (the
legacy **envelope**: `rc` is S or E, `err_cd`, `msg`, `data`). We never look inside a service.

In [3]:
import httpx

from bahi import ERP
from bahi.erp.receipts import install_receipts
from bahi.server import serve

erp = ERP()
install_receipts(erp)          # goods receipt (GRN), the last step of procure-to-pay
url = serve(port=8077, erp=erp)

login = httpx.post(url + "/login", json={"user_id": "KJOSHI", "password": "kavita@123", "ou_cd": "RSW-PUN-01"}).json()
H = {"Authorization": "Bearer " + login["token"]}
print(login["msg"])


def svc(name, **payload):
    """Call one Bahi service over HTTP, as Kavita. Returns the envelope."""
    return httpx.post(f"{url}/svc/{name}", json=payload, headers=H).json()


show_json(svc("stk_query", itm_cd="GSK-SIL-05L", whs_cd="WH-PUN-STR"))

Bahi is running at http://127.0.0.1:8077 · legacy client: http://127.0.0.1:8077/legacy
WELCOME, KJOSHI. LAST SIGN-IN 24-10-2026 18:02
{
 "rc": "S",
 "err_cd": null,
 "msg": "",
 "data": {
  "itm_cd": "GSK-SIL-05L",
  "whs_cd": "WH-PUN-STR",
  "on_hand": 54000,
  "allocated": 0,
  "on_order": 0,
  "avg_daily_issue": 6000,
  "reorder_lvl": 30000,
  "cover_days": 9
 }
}


In [4]:
env = svc("itm_search", q="gasket 5")
print(env["msg"])
pd.DataFrame(env["data"]["rows"])

4 ROWS FOUND


,itm_cd,descr,uom_cd,active
0,GSK-SIL-05L,"Gasket, silicone, 5 L",NOS,1
1,GSK-SIL-05L-R,"Gasket, silicone, 5 L (rework)",NOS,1
2,GSK-RUB-05L,"Gasket, rubber, 5 L",NOS,0
3,GSK-SIL-07L,"Gasket, silicone, 7.5 L",NOS,1


Four gaskets look alike. `-R` is rework stock and is never ordered; the rubber one is inactive. A person has to
know that. Bahi has 38 services like these, plus 3 for goods receipt:

In [5]:
cat = httpx.get(url + "/services").json()
print(len(cat), "services")
pd.DataFrame([{k: s[k] for k in ("name", "kind", "doc")} for s in cat]).head(12)

41 services


,name,kind,doc
0,itm_search,read,ITEM ENQUIRY
1,itm_get,read,ITEM MASTER GET
2,sup_search,read,SUPPLIER ENQUIRY
3,sup_get,read,SUPPLIER MASTER GET
4,sup_create,write,SUPPLIER CREATE
5,sup_cert_get,read,SUPPLIER CERT LIST
6,sup_cert_upload,write,SUPPLIER CERT UPLOAD
7,sup_rate_get,read,RATE CONTRACT GET
8,cc_list,read,COST CENTRE LIST
9,cc_get,read,COST CENTRE GET


### TODO (exercise 1): which Deccan?
Call the supplier search service `sup_search` with `q="Deccan"`. How many rows come back, and which one must
never be used? Put its code in `blocked`.

In [6]:
SOLUTION = True
blocked = None
if SOLUTION:
    rows = svc("sup_search", q="Deccan")["data"]["rows"]
    display(pd.DataFrame(rows))
    blocked = next(r["sup_cd"] for r in rows if r["status"] == "BLOCKED")
print("blocked duplicate:", blocked)
assert blocked == "DP-RJK-0009"

,sup_cd,name,city,state,status
0,DP-RJK-0042,Deccan Polymers,Rajkot,GJ,ACTIVE
1,DP-RJK-0009,"Deccan Polymers (old code, do not use)",Rajkot,GJ,BLOCKED
2,DPI-AMD-0107,Deccan Paints & Inks,Ahmedabad,GJ,ACTIVE


blocked duplicate: DP-RJK-0009


## 2. The Before, by code

`Desk` is one person at one PC with the legacy client open. It reads the same screen files the client reads
(`erp/screens/*.yaml`), so pressing a button here makes exactly the service call the client would make. `Tally`
counts what the person did: screens opened, fields typed, codes typed, buttons, sign-ins, error codes.

Here is the order-entry screen as a file: its fields, and which service each button calls.

In [7]:
import pathlib

import yaml

import bahi
from bahi.legacy_ui.driver import Desk, Tally, screen_yaml

print(screen_yaml("PUR-PO-01")[:1500])

code: PUR-PO-01
title: PURCHASE ORDER ENTRY
menu: Purchase > Order entry
roles:
- PUR-EXEC
- PUR-SR
- SYS-ADMIN
help: 'Enter the source request, supplier and item, then press Fetch: it brings in
  the contract rate, tax code, payment terms and the request''s cost centre, date
  and quantity. Save gives the order number. Orders without a request (type DIR) are
  for Rs 50,000 or less.'
fields:
- name: po_no
  label: PO NO
  type: str
  readonly: true
- name: po_type
  label: PO TYPE
  type: str
  req: true
  default: STD
  help: STD/DIR
- name: pr_no
  label: PR NO
  type: str
  req: false
  lookup: pr_search
- name: sup_cd
  label: SUP CD
  type: str
  req: true
  lookup: sup_search
- name: itm_cd
  label: ITEM CD
  type: str
  req: true
  lookup: itm_search
- name: qty
  label: QTY
  type: int
  req: true
- name: rate
  label: RATE
  type: float
  req: true
- name: tax_cd
  label: TAX CD
  type: str
  req: true
  lookup: tax_list
- name: pt_cd
  label: PAY TERM
  type: str
  readonly:

Now the five missions, the way someone does them the first time: keeping what **Fetch** fills in.

In [8]:
tally = Tally()
http = httpx.Client(base_url=url)
AMEND = pathlib.Path(bahi.__file__).parent / "experience" / "screens" / "PUR-POM-06.yaml"


def desk(who, uid, pw):
    d = Desk(http, tally, who)
    if AMEND.exists():  # the order-amendment screen the Experience serves
        d.screens["PUR-POM-06"] = yaml.safe_load(AMEND.read_text())
    return d.sign_in(uid, pw)


marks = {}
def mission(name):
    marks[name] = dict(tally.c)


mission("1 stock")
kavita = desk("Kavita", "KJOSHI", "kavita@123")
s = kavita.open("INV-ITM-03").type(q="gasket 5").press("Search").pick(0)
item = s.form["itm_cd"]
s = kavita.open("INV-STK-02").type(itm_cd=item).press("Query")
print("stock:", s.form["on_hand"], "on hand,", s.form["cover_days"], "days of cover (one warehouse)")

mission("2 request")
s = kavita.open("PUR-PR-01").type(itm_cd=item).press("Fetch")
print("Fetch filled cost centre:", s.form["cc_cd"])
s.type(qty="20000", need_dt="09-11-2026", remarks="Diwali run").press("Save").press("Submit")
pr_no = s.form["pr_no"]
print(s.status)

mission("3 approve")
rohit = desk("Rohit", "RMEHTA", "rohit@123")
s = rohit.open("PUR-PR-04").press("Inbox")
s.pick(next(i for i, r in enumerate(s.rows) if r["doc_no"] == pr_no)).press("Fetch").type(decision="A").press("Confirm")
print(s.status)

mission("4 order")
s = kavita.open("PUR-POH-02").type(itm_cd=item).press("Query").pick(0)
sup = s.form["sup_cd"]
s = kavita.open("PUR-PO-01").type(pr_no=pr_no, sup_cd=sup, itm_cd=item).press("Fetch").press("Save")
print("Save:", s.status)
s.type(tax_cd="GST18-INTER").press("Save")
po_no = s.form["po_no"]
print("Save again:", s.status, "| status field:", s.form["status"])
s = kavita.open("PUR-POR-05").type(po_no=po_no).press("Fetch").press("Release")
print("Release:", s.status)

stock: 54000 on hand, 9 days of cover (one warehouse)
Fetch filled cost centre: PRD-05
PR/2026/04417 SUBMITTED FOR APPROVAL


PR/2026/04417 APPROVED BY RMEHTA
Save: ERR-PO-0412: TAX REGION DOES NOT MATCH SUPPLIER STATE (GJ)
Save again: PO/2026/01182 SAVED | status field: DR
Release: ERR-PO-0715: DOCUMENT NOT IN RELEASABLE STATUS (DR)


**Saved is not done.** The status bar said SAVED; the status field says `DR`, a Draft. The order took cost
centre PRD-05 from the request, and PRD-05 closed on 15 October. Release refuses it. The fix is order amendment:
change the cost centre to PRD-07, then release again. Then Neha receives the goods.

In [9]:
if AMEND.exists():
    s = kavita.open("PUR-POM-06").type(po_no=po_no).press("Fetch").type(cc_cd="PRD-07").press("Amend")
else:  # older installs without the amendment screen: the same service, called directly
    print(svc("po_amend", po_no=po_no, cc_cd="PRD-07"))
s = kavita.open("PUR-POR-05").type(po_no=po_no).press("Fetch").press("Release")
print("Release:", s.status)

mission("5 receive")
neha = desk("Neha", "NDESAI", "neha@123")
for q in (12000, 8000):
    s = neha.open("INV-GRN-03").type(po_no=po_no).press("Fetch").type(qty=str(q)).press("Save")
    print(s.status, "| balance", s.form.get("balance"))
marks["end"] = dict(tally.c)

{'rc': 'S', 'err_cd': None, 'msg': 'PO/2026/01182 AMENDED', 'data': {'po_no': 'PO/2026/01182', 'status': 'AP'}}
Release: PO/2026/01182 RELEASED


GRN/2026/30001 SAVED | balance 8000
GRN/2026/30002 SAVED | balance 0


In [10]:
names = list(marks)
cols = ["screens opened", "fields typed", "codes typed", "characters typed", "buttons pressed", "sign-ins", "error codes"]
before = pd.DataFrame([{**{"mission": names[i]}, **{c: marks[names[i + 1]].get(c, 0) - marks[names[i]].get(c, 0) for c in cols}}
                       for i in range(len(names) - 1)]).set_index("mission")
before.loc["total"] = before.sum()
before

,screens opened,fields typed,codes typed,characters typed,buttons pressed,sign-ins,error codes
mission,,,,,,,
1 stock,3,4,1,35,4,1,0
2 request,1,4,1,36,3,0,0
3 approve,2,3,1,16,5,1,0
4 order,4,7,7,83,9,0,2
5 receive,3,6,2,49,5,1,0
total,13,24,12,219,26,3,2


### TODO (exercise 2): the careful path
Suresh knows every trap: he types `PRD-07` on the request and `GST18-INTER` on the order before the first Save.
Count what *that* costs. Re-run missions 2 and 4 his way on a fresh Rasoi and fill `careful_codes` with the number
of codes typed in mission 4. (Knowing the traps removes errors. Does it remove steps?)

In [11]:
SOLUTION = True
careful_codes = None
if SOLUTION:
    erp2 = ERP(); install_receipts(erp2)
    from fastapi.testclient import TestClient
    from bahi.server import create_app
    app2 = create_app(); app2.state.tenants["default"] = erp2
    t2 = Tally(); http2 = TestClient(app2)
    k2 = Desk(http2, t2, "Kavita").sign_in("KJOSHI", "kavita@123")
    s = k2.open("PUR-PR-01").type(itm_cd="GSK-SIL-05L").press("Fetch").type(qty="20000", need_dt="09-11-2026", cc_cd="PRD-07").press("Save").press("Submit")
    pr2 = s.form["pr_no"]
    r2 = Desk(http2, t2, "Rohit").sign_in("RMEHTA", "rohit@123")
    s = r2.open("PUR-PR-04").press("Inbox")
    s.pick(next(i for i, r in enumerate(s.rows) if r["doc_no"] == pr2)).press("Fetch").type(decision="A").press("Confirm")
    c0 = t2.c["codes typed"]
    s = k2.open("PUR-POH-02").type(itm_cd="GSK-SIL-05L").press("Query").pick(0)
    s = k2.open("PUR-PO-01").type(pr_no=pr2, sup_cd=s.form["sup_cd"], itm_cd="GSK-SIL-05L").press("Fetch").type(tax_cd="GST18-INTER").press("Save")
    print("careful Save:", s.status, "| status", s.form.get("status"))
    s = k2.open("PUR-POR-05").type(po_no=s.form["po_no"]).press("Fetch").press("Release")
    print("careful Release:", s.status)
    careful_codes = t2.c["codes typed"] - c0
print("codes typed in mission 4, careful path:", careful_codes, "| first time:", before.loc["4 order", "codes typed"])

careful Save: PO/2026/01182 SAVED | status AP
careful Release: PO/2026/01182 RELEASED
codes typed in mission 4, careful path: 6 | first time: 7


## 3. The After: Ask Rasoi

A new, fresh Rasoi, and the same five missions, one sentence each. Each person gets their own **Ask Rasoi**,
signed in as themselves. When Ask Rasoi asks a question ("Shall I go ahead?"), the person answers with fixed
rules (`bahi.course.l1.answer`), so the run replays exactly.

For now, treat Ask Rasoi as a black box, like Bahi: a sentence goes in, a reply comes out.

In [12]:
from bahi.course.l1 import MISSIONS, play_missions

erp3 = ERP()
agents = {}
log = play_missions(erp3, agents=agents)

for m in log:
    display(Markdown(f"### {m['title']} ({m['who']}) · done: **{m['done']}**"))
    for t in m["turns"]:
        display(Markdown(f"> **{m['who']}:** {t['user']}\n\n**Ask Rasoi:** " + t["reply"].replace("\n", "  \n")))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

### Check the stock (Kavita) · done: **True**

...

### Receive the rest (Neha) · done: **True**

> **Neha:** The other 8,000 have come in too.

**Ask Rasoi:** Here is the preview to record the goods receipt for the remaining balance:  
  
- **PO:** PO/2026/01182 (Deccan Polymers)  
- **Item:** GSK-SIL-05L (Receiving: 8,000 | Remaining balance: 0, order will close)  
- **Warehouse:** WH-PUN-STR  
  
Shall I go ahead and record this receipt?

> **Neha:** Yes, go ahead.

**Ask Rasoi:** Goods receipt **GRN/2026/30002** has been recorded for the remaining 8,000 gaskets against PO/2026/01182.   
  
The purchase order is now fully received and closed. Stock for GSK-SIL-05L stands at 74,000 units.

### A cheaper supplier? (Kavita) · done: **True**

> **Kavita:** Shakti Seals in Vadodara quotes Rs 10.40 for the same gasket. Can we buy from them instead?

**Ask Rasoi:** No, we cannot order from Shakti Seals right now:  
  
1. **Food-grade certification required:** Silicone gaskets are food-contact items and, per policy (POL-PUR-001 §6.2, POL-SUP-003 §4.1), can only be bought from suppliers holding a verified food-grade certificate on file in Bahi.  
2. **Not registered:** Shakti Seals is not registered in Bahi. They would first need to be onboarded and have their food-grade certificate verified.  
3. **Existing order released:** PO/2026/01182 has already been released to Deccan Polymers.

Bahi, read back after the missions: the same document numbers as the Before, and nothing silent.

In [13]:
display(pd.DataFrame(erp3.q("SELECT pr_no, status, cc_cd, est_value, approved_by FROM pr_hdr WHERE raised_on >= '2026-10-26'")))
display(pd.DataFrame(erp3.q("SELECT po_no, status, cc_cd, tax_cd, gross_value, approved_by, warn_cd FROM po_hdr WHERE created_on >= '2026-10-26'")))
display(pd.DataFrame(erp3.q("SELECT h.grn_no, h.po_no, l.qty, h.received_by FROM grn_hdr h JOIN grn_line l ON l.grn_no = h.grn_no")))
print("Shakti Seals registered?", erp3.q("SELECT sup_cd FROM supplier WHERE name LIKE '%Shakti%'") or "no")

,pr_no,status,cc_cd,est_value,approved_by
0,PR/2026/04417,CL,PRD-07,269040.0,RMEHTA


,po_no,status,cc_cd,tax_cd,gross_value,approved_by,warn_cd
0,PO/2026/01182,CL,PRD-07,GST18-INTER,280840.0,RMEHTA,None


,grn_no,po_no,qty,received_by
0,GRN/2026/30001,PO/2026/01182,12000,NDESAI
1,GRN/2026/30002,PO/2026/01182,8000,NDESAI


Shakti Seals registered? no


## 4. How it works

Now open the black box one level. Ask Rasoi is three things:
- a **toolbox**: one tool per business activity, each a folder with `tool.json` and `tool.py`;
- **MCP**, the plug: Ask Rasoi asks the toolbox server what tools it has, then calls them through it;
- an **orchestrator LLM** that reads the request, picks the next tool, reads the result, and decides again.

Here is what went over the plug when Kavita's Ask Rasoi connected: `tools/list`, then the first `tools/call`.

In [14]:
kav = agents["KJOSHI"]
print("tools discovered over MCP:", len(kav.tools.names))
print(kav.tools.names)
w = kav.client.wire
print("\nfirst message on the wire:", w[0]["method"])
show_json({"method": w[1]["method"], "params": w[1]["params"]}, 400)

tools discovered over MCP: 15
['find_item', 'check_stock', 'find_supplier', 'check_budget', 'search_policy', 'create_purchase_request', 'list_my_approvals', 'approve_purchase_request', 'create_purchase_order', 'approve_purchase_order', 'release_purchase_order', 'order_status', 'receive_goods', 'register_supplier', 'send_message']

first message on the wire: tools/list
{
 "method": "tools/call",
 "params": {
  "name": "find_item",
  "arguments": {
   "query": "5 litre silicone gasket"
  }
 }
}


Every tool call Ask Rasoi made, as the "How it works" cards on /after show them:

In [15]:
cards = pd.DataFrame([{"mission": m["title"], "who": m["who"], "tool": c["tool"],
                       "arguments": json.dumps(c["args"])[:90],
                       "ok": c["result"].get("ok") if isinstance(c["result"], dict) else None}
                      for m in log for c in m["cards"]])
cards

,mission,who,tool,arguments,ok
0,Check the stock,Kavita,find_item,"{""query"": ""5 litre silicone gasket""}",True
1,Check the stock,Kavita,check_stock,"{""item_code"": ""GSK-SIL-05L""}",True
2,Raise the purchase request,Kavita,check_budget,{},True
3,Raise the purchase request,Kavita,create_purchase_request,"{""item_code"": ""GSK-SIL-05L"", ""confirm"": false, ""quantity"": 20000, ""need_by"":...",True
4,Raise the purchase request,Kavita,create_purchase_request,"{""item_code"": ""GSK-SIL-05L"", ""cost_centre"": ""PRD-07"", ""need_by"": ""2026-11-09...",True
5,"Approve it, as Rohit",Rohit,list_my_approvals,{},True
6,"Approve it, as Rohit",Rohit,order_status,"{""request_no"": ""PR/2026/04417""}",True
7,"Approve it, as Rohit",Rohit,approve_purchase_request,"{""request_no"": ""PR/2026/04417"", ""decision"": ""approve""}",True
8,Make and release the order,Kavita,order_status,"{""request_no"": ""PR/2026/04417""}",True
9,Make and release the order,Kavita,find_supplier,"{""name"": ""Deccan Polymers""}",True


### One tool, opened
`tool.json` is what the LLM reads: a name, a description, and the **JSON schema** of its input and output.
`tool.py` is what runs: ordinary Python that calls Bahi's services and returns a receipt.

In [16]:
from bahi.toolbox import SPECS, Toolbox

po_tool = SPECS["create_purchase_order"]
print(json.dumps({k: po_tool.spec[k] for k in ("name", "description", "input_schema")}, indent=1)[:1600])

{
 "name": "create_purchase_order",
 "description": "Create a purchase order (PO) with a supplier. Give request_no for an approved purchase request (it supplies item, quantity, cost centre and date), or item_code, quantity, cost_centre and need_by for a small direct order up to Rs 50,000. confirm=false previews price, tax and value; confirm=true saves it. Returns a receipt with the order's real status: an APPROVED order still has to be released.",
 "input_schema": {
  "type": "object",
  "properties": {
   "supplier_code": {
    "type": "string",
    "description": "Bahi supplier code, from find_supplier"
   },
   "request_no": {
    "type": "string",
    "description": "an APPROVED purchase request"
   },
   "item_code": {
    "type": "string",
    "description": "for a direct order"
   },
   "quantity": {
    "type": "integer",
    "description": "for a direct order"
   },
   "cost_centre": {
    "type": "string",
    "description": "for a direct order"
   },
   "need_by": {
    "typ

In [17]:
print(po_tool.source)

"""Create purchase order: Create a purchase order (PO) with a supplier.

Input and output are described in tool.json next to this file."""
from bahi.agent.tools import ServiceTimeout, ToolError, _bahi, _receipt, _tax_code, content_key


def run(s, supplier_code, request_no=None, item_code=None, quantity=None, cost_centre=None, need_by=None,
        rate=None, confirm=False):
    if request_no:  # the request carries what to buy, how many, who pays and by when
        pr = _bahi(s, "pr_get", pr_no=request_no)
        item_code, quantity = pr["lines"][0]["itm_cd"], pr["lines"][0]["qty"]
        cost_centre, need_by = pr["cc_cd"], pr["need_dt"]
    elif not all([item_code, quantity, cost_centre, need_by]):
        raise ToolError("A direct order needs item_code, quantity, cost_centre and need_by.",
                        "Ask the user, or raise a purchase request first.")
    key = content_key("PO", s.user_id, supplier_code, item_code, quantity, cost_centre, need_by)
    found = _bahi(s,

### TODO (exercise 3): find the traps in the code
Two lines of `tool.py` remove two traps you hit by hand: the wrong tax code (ERR-PO-0412) and the silent Draft
on a closed cost centre. Put those two lines (as text, stripped) in `tax_line` and `draft_line`.

In [18]:
SOLUTION = True
tax_line = draft_line = None
if SOLUTION:
    lines = [l.strip() for l in po_tool.source.splitlines()]
    tax_line = next(l for l in lines if l.startswith("tax_cd = _tax_code"))
    draft_line = next(l for l in lines if l.startswith('if budget["cc_state"] == "CLOSED"'))
print(tax_line)
print(draft_line)

tax_cd = _tax_code(s, supplier_code)  # from the supplier's state, not Bahi's default (the ERR-PO-0412 trap)
if budget["cc_state"] == "CLOSED":


### TODO (exercise 4): call a tool yourself
You do not need an LLM to use a tool. `Toolbox(erp, user)` binds the tools to one person's Bahi session.
As Kavita, on a fresh Rasoi, call `check_budget` for `PRD-05`, then try a **direct** order (no request) for
20,000 gaskets with `create_purchase_order`, `confirm=False`. What does each say, and what is the next step?

In [19]:
SOLUTION = True
budget = direct = None
if SOLUTION:
    tb = Toolbox(ERP(), "KJOSHI")
    budget = tb.call("check_budget", {"cost_centre": "PRD-05"})
    direct = tb.call("create_purchase_order", {"supplier_code": "DP-RJK-0042", "item_code": "GSK-SIL-05L",
                                               "quantity": 20000, "cost_centre": "PRD-07", "need_by": "2026-11-09"})
show_json(budget, 500)
show_json(direct, 500)

{
 "ok": true,
 "code": "PRD-05",
 "name": "5-litre line, regular production",
 "open": false,
 "closed_on": "2026-10-15",
 "budget_left": 190000.0,
 "note": "PRD-05 is CLOSED. Bahi will silently save any order on it as a DRAFT that never reaches the supplier. Ask the user for an open cost centre."
}
{
 "ok": false,
 "error": "A direct order is limited to Rs 50,000; this one is Rs 280,840.",
 "next_step": "Raise a purchase request first with create_purchase_request.",
 "bahi_code": ""
}


Compare with the Before: Bahi said `ERR-PO-0610: DIRECT PO LIMIT 50000 EXCEEDED, RAISE PR`. The tool says what
went wrong in words and gives a `next_step`.

## 5. The Before and the After, side by side

In [20]:
A = [m for m in log if m["id"] != "shakti"]
after = {"screens opened": 0, "fields typed": 0,
         "codes typed": sum("PO/" in t["user"] for m in A for t in m["turns"]),
         "characters typed": sum(len(t["user"]) for m in A for t in m["turns"]),
         "buttons pressed": 0, "sign-ins": 0, "error codes": 0}
score = pd.DataFrame({"Before (first time)": before.loc["total"], "After (Ask Rasoi)": pd.Series(after)})
score.loc["messages typed"] = [None, sum(len(m["turns"]) for m in A)]
score.loc["tool calls (Ask Rasoi's work)"] = [None, sum(len(m["cards"]) for m in A)]
score

,Before (first time),After (Ask Rasoi)
screens opened,13.0,0
fields typed,24.0,0
codes typed,12.0,1
characters typed,219.0,474
buttons pressed,26.0,0
sign-ins,3.0,0
error codes,2.0,0
messages typed,NaN,12
tool calls (Ask Rasoi's work),NaN,17


### TODO (exercise 5): where did the effort go?
The After types *more* characters than the Before. Work out the share of the After's messages that were just
"Yes, go ahead." and put it in `yes_share`. Then say in one sentence what replaced the codes.

In [21]:
SOLUTION = True
yes_share = None
if SOLUTION:
    msgs = [t["user"] for m in A for t in m["turns"]]
    yes_share = sum(x.startswith("Yes") for x in msgs) / len(msgs)
print(f"{yes_share:.0%} of the messages were a yes to a preview")

50% of the messages were a yes to a preview


## 6. Your module (assignment A1)

Every student takes the same Before and After to one more corner of Rasoi. Eight module kits ship with Bahi;
each has three legacy screens, its traps, and one sentence its After should handle.

In [22]:
import importlib

KITS = ["pur_amend", "fin_payables", "hr_leave", "sales_orders", "svc_warranty", "inv_transfers", "travel_claims", "legal_contracts"]
rows = []
for k in KITS:
    kit = importlib.import_module(f"bahi.modules.{k}").KIT
    rows.append({"kit": k, "title": kit["title"], "speaker": kit["speaker"], "screens": ", ".join(kit["screens"]),
                 "the After, in one sentence": kit["after"]})
pd.DataFrame(rows)

,kit,title,speaker,screens,"the After, in one sentence"
0,pur_amend,Purchasing · amend PO,KJOSHI,"PUR-POL-07, PUR-POM-08, PUR-POA-09","Push Deccan's gasket delivery to 9 November and cut it to 15,000."
1,fin_payables,Finance · payables,KJOSHI,"FIN-AP-01, FIN-AP-03, FIN-AP-05","Why is Deccan's invoice 4471 on hold, and can we pay it Friday?"
2,hr_leave,HR · leave,KJOSHI,"HR-LV-01, HR-LV-02, HR-LV-04",I need Thursday and Friday off for my sister's wedding.
3,sales_orders,Sales · orders,ARAO,"SAL-CUS-01, SAL-QTN-02, SAL-ORD-03",Quote Annapurna Stores 500 five-litre cookers at dealer price.
4,svc_warranty,Service · warranty,FSHAIKH,"SVC-SER-01, SVC-TKT-02, SVC-WAV-03",A customer's whistle broke after three weeks; replace it.
5,inv_transfers,Inventory · transfers,NDESAI,"INV-STK-07, INV-TRN-04, INV-GTP-05","Move 2,000 gaskets from Chakan to the Pune line tonight."
6,travel_claims,Travel · claims,KJOSHI,"TRV-REQ-01, TRV-BKG-02, TRV-CLM-03","Book me Pune to Rajkot on Tuesday to visit Deccan, back Wednesday."
7,legal_contracts,Legal · contracts,KJOSHI,"LEG-REQ-01, LEG-REV-02, LEG-REG-03","Raise an NDA with Shakti Seals on our paper, valid two years."


### TODO (exercise 6): your module's Before
Pick your kit. Install it (`ERP(modules=[...])`), print its screens' YAML, and run its scripted legacy path
(`happy_path`) and its traps. These go into A1.

In [23]:
SOLUTION = True
MY_KIT = "hr_leave"   # change to your kit
if SOLUTION:
    erp_m = ERP(modules=[MY_KIT])
    mod = importlib.import_module(f"bahi.modules.{MY_KIT}")
    from bahi.server import load_screens
    for code in mod.KIT["screens"]:
        sc = load_screens([MY_KIT])[code]
        print(code, "·", sc["title"], "·", len(sc.get("fields", [])), "fields ·", [t["button"] for t in sc.get("tasks", [])])
    legacy = importlib.import_module(f"bahi.modules.{MY_KIT}.legacy")
    print("\nthe careful legacy path:", legacy.happy_path(erp_m), "·", len(erp_m.calls), "service calls")
    print("\nits traps (each on a fresh Rasoi):")
    for name, trap in legacy.TRAPS.items():
        good, what = trap(ERP(modules=[MY_KIT]))
        print(f"  {'yes' if good else 'NO '}  {name}: {what}"[:160])

HR-LV-01 · LEAVE BALANCE ENQUIRY · 6 fields · ['Find Emp', 'Show', 'Holidays']


HR-LV-02 · LEAVE APPLICATION · 9 fields · ['Fetch', 'Save', 'Cancel Leave']
HR-LV-04 · LEAVE APPROVAL · 5 fields · ['Inbox', 'Decide', 'Clear Held']

the careful legacy path: {'doc': 'LV/2026/00388', 'status': 'AP', 'ok': True} · 7 service calls

its traps (each on a fresh Rasoi):
  yes  silent: CL short, saved as HD with 1 day unpaid: rc=S msg='LV/2026/00388 SAVED' status=HD lwp=1.0 routed=False
  yes  variant: half day on two dates refused: ERR-LV-0611 HALF DAY ONLY FOR SINGLE DATE
  yes  variant: comp-off needs the worked date: no ref_dt: ERR-LV-0620; with ref_dt: LV/2026/00388 SAVED
  yes  screen_state: Save without Fetch: ERR-SCR-0221 FETCH NOT DONE
  yes  cascade: approval posts the balance: EL availed 6.0→8.0, pending 0.0→2.0→0.0
  yes  cascade: roster and payroll LWP feed: roster [('2026-10-29', 'CL'), ('2026-10-30', 'LWP')], payroll LWP feed [1.0]
  yes  shared_engine: sp_appr_route, visible in the purchasing inbox: sp_appr_route in save: True; in Rohit's purchasing inbox: Tru

## What you did
- Bahi's services are black boxes: an input goes in, an envelope comes out.
- The Before, counted: screens, fields, codes, sign-ins and the traps (ERR-PO-0412, the silent Draft, ERR-PO-0715).
- The After: one sentence per mission, a preview and a yes before every save, receipts read back from Bahi.
- How it works: one tool per activity (`tool.json` + `tool.py`), discovered over MCP, chosen by an orchestrator LLM.

**Next time (Lecture 2, The Brain):** we take the tools away and look inside the LLM itself.